In [1]:
### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt
import corner

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="0"

physical_devices = tf.config.list_physical_devices("GPU") 

tf.config.experimental.set_memory_growth(physical_devices[0], True)

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

modelpath = '/home/hatte/M4/models'

2025-11-03 14:02:16.224119: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762178536.236589   62651 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762178536.240309   62651 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-03 14:02:16.255343: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Current GPU usage:
 - GPU0: 0B



I0000 00:00:1762178537.618780   62651 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15357 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:41:00.0, compute capability: 8.6


In [2]:
def scheduler(epoch, lr,):
    ## Learning rate scheduler
    # Decreases learning rate in-training for stability
    if lr < 1e-5:
        return float(lr)
    else:
        return float(lr * tf.math.exp(-0.000006))

In [3]:
df_full = pd.read_hdf('../grids/Chiara-dnufit.hdf5', key='df') ## edit for your grid!!

df_full['logLPhot'] = np.log10(df_full['LPhot'])

df_full['lognumax'] = np.log10(df_full['numax']*3090)

df_full['logdnufit'] = np.log10(df_full['dnufit'])

df_full['logAge'] = np.log10(df_full['age'])

df_full['logTeff'] = np.log10(df_full['Teff'])

df_full['logmassini'] = np.log10(df_full['massini'])

df_full['logyini'] = np.log10(df_full['yini'])

df_full['logalphaMLT'] = np.log10(df_full['alphaMLT'])


#### define inputs
inputs = ['logmassini', 'zini', 'yini', 'alphaMLT', 'logAge', 'eta', 'alphaFe']

#### define outputs
classical_outputs = ['FeH', 'logLPhot', 'Teff']
astero_outputs = ['lognumax', 'logdnufit'] 

outputs = classical_outputs+astero_outputs

df = df_full[inputs+outputs]

df_norm = (df - df.min())/(df.max() - df.min())

## check df_norm.describe looks reasonable (min=0, max=1):
df_norm.describe()

#### train/test split with seed 
seed = 42

df_train = df_norm.sample(frac=0.95, random_state=seed)
df_test = df_norm.drop(df_train.index)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.05, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))
print("Test set: ", len(df_test))

Training set:  6754081
Validation set:  355478
Test set:  374187


In [4]:
#unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'numax':0.001/3090, 'dnuSer':0.0001/135}

unnormed_weights_dict = {'FeH':0.01, 'logLPhot':0.001, 'Teff':10, 'lognumax':0.001, 'logdnufit':0.0001}

unnormed_weights = list(unnormed_weights_dict.values())

weights = [2*unnormed_weights_dict[i]/(df[i].max() - df[i].min()) for i in outputs]

In [32]:
n_dense_layers = 6

dense_layer_units = 128

Nepochs = 400000

learning_rate = 0.0001

model_name = 'M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam'

loss_func = 'WMSE'

df_train_inputs.join(df_train_outputs).to_hdf(f'{modelpath}/long-runs/training-data/training-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.hdf5', key = 'df')

In [33]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

if not os.path.exists(full_model_dir):
    os.makedirs(full_model_dir)

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, )

In [13]:
######## map out model architecture
#### input layer
nn_input = keras.Input(shape=(len(inputs),))

#### dense layer(s)
for n_dense_layer in range(n_dense_layers):
    if n_dense_layer == 0:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(nn_input)
    else:
        dense_layer = layers.Dense(dense_layer_units, activation='relu')(dense_layer)

#### output layer
nn_output =  layers.Dense(len(outputs), activation='linear')(dense_layer)

######## store architecture as keras model
model = keras.Model(inputs=nn_input, outputs=nn_output, name=model_name)

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate))

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=Nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback]) 

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 1/400000
131/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 983852.0625 

KeyboardInterrupt: 

In [25]:
custom_objects =  {'WMSE':WMSE_metric}
model= tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)

In [9]:
n_learning_rate = model.optimizer.get_config()['learning_rate']

Nepochs = 277171

nepochs = 400000

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

model.compile(loss=WMSE(weights), optimizer=tf.keras.optimizers.Adam(learning_rate=n_learning_rate))

In [ ]:
history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=nepochs,
          shuffle=True, callbacks = [tb_callback, cp_callback, lr_callback],
          initial_epoch=Nepochs)

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)

Epoch 277172/400000


I0000 00:00:1761832521.377408 4162794 service.cc:148] XLA service 0x7eccc0002310 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1761832521.377594 4162794 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A4500, Compute Capability 8.6
2025-10-30 13:55:21.415544: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1761832521.540194 4162794 cuda_dnn.cc:529] Loaded cuDNN version 90300
2025-10-30 13:55:21.620821: W external/local_xla/xla/service/gpu/nvptx_compiler.cc:930] The NVIDIA driver's CUDA version is 12.2 which is older than the PTX compiler version 12.5.82. Because the driver is older than the PTX compiler version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.
2025-10-30 13:55:22.11793

 33/413 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4155.6421

I0000 00:00:1761832523.642356 4162794 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


397/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 935.6887

2025-10-30 13:55:25.716924: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_258', 32 bytes spill stores, 32 bytes spill loads

2025-10-30 13:55:25.800872: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_446', 24 bytes spill stores, 24 bytes spill loads

2025-10-30 13:55:25.922746: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_446_0', 36 bytes spill stores, 36 bytes spill loads

2025-10-30 13:55:25.949674: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_446', 136 bytes spill stores, 136 bytes spill loads

2025-10-30 13:55:26.006537: I external/local_xla/xla/stream_

413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 911.6781

2025-10-30 13:55:28.272952: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30', 32 bytes spill stores, 32 bytes spill loads

2025-10-30 13:55:28.346350: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_30_0', 168 bytes spill stores, 168 bytes spill loads




Epoch 277172: val_loss improved from inf to 133.71507, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-400000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 910.2316 - val_loss: 133.7151 - learning_rate: 1.9670e-05
Epoch 277173/400000
397/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 160.1246
Epoch 277173: val_loss improved from 133.71507 to 129.81544, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam-nlayers-6-nunits-128-epochs-400000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 160.6683 - val_loss: 129.8154 - learning_rate: 1.9669e-05
Epoch 277174/400000
407/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 186.5578
Epoch 277174: val_loss did not improve from 129.81544
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 186.6169 - val_loss: 448.9240 - l

In [34]:
custom_objects =  {'WMSE':WMSE_metric}

model= tf.keras.models.load_model(checkpoint_dir, custom_objects = custom_objects)

n_learning_rate = model.optimizer.get_config()['learning_rate']

model_name = model_name + '-SGD'

print(model_name)

M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam-SGD


In [11]:
n_learning_rate

9.999995199905243e-06

In [ ]:
checkpoint_dir = f'{modelpath}/long-runs/checkpoint/chk-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.model.keras'

full_model_dir = f'{modelpath}/long-runs/full-model/mod-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}/'

tb_callback = tf.keras.callbacks.TensorBoard(log_dir = f'{modelpath}/logs/long-runs/log-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}')

historyfile = f'{modelpath}/long-runs/history/hist-{model_name}-nlayers-{n_dense_layers}-nunits-{dense_layer_units}-epochs-{Nepochs}-lrate-{learning_rate}-lossfunc-{loss_func}.json'
    
cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath = checkpoint_dir, verbose = 1, save_best_only = True, save_freq = 'epoch')

from tensorflow.keras.mixed_precision import LossScaleOptimizer, global_policy

tf.keras.backend.clear_session()

sgd = tf.keras.optimizers.SGD(learning_rate = 1e-6, momentum=0.0, clipnorm = 1, nesterov = True)

if global_policy().name.startswith("mixed"):

    sgd = LossScaleOptimizer(sgd)

model.compile(loss=WMSE(weights), optimizer=sgd)

history = model.fit(df_train_inputs,
          df_train_outputs,
          validation_data=(df_val_inputs,df_val_outputs),
          batch_size=2**14, #change higher
          verbose=1,
          epochs=450000,
          shuffle=True, callbacks = [tb_callback, cp_callback], initial_epoch = 400000) 


Epoch 400001/450000
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 102.8439
Epoch 400001: val_loss improved from inf to 104.10930, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam-SGD-nlayers-6-nunits-128-epochs-400000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 102.8438 - val_loss: 104.1093
Epoch 400002/450000
405/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 102.7158
Epoch 400002: val_loss improved from 104.10930 to 103.47070, saving model to /home/hatte/M4/models/long-runs/checkpoint/chk-M4-log-mass-age-dnu-numax-Lphot-exponent-6e-6-Adam-SGD-nlayers-6-nunits-128-epochs-400000-lrate-0.0001-lossfunc-WMSE.model.keras
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - loss: 102.7171 - val_loss: 103.4707
Epoch 400003/450000
396/413 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 102.9415
Epoch 400003: val_loss did not improve from 103.47070
413/413 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 102.9

[<Variable path=dense/kernel, shape=(7, 128), dtype=float32, value=[[-1.98153034e-01 -2.39497676e-01 -2.50401646e-01  2.30925485e-01
   -1.47562280e-01 -4.63613868e-02 -2.00637832e-01  2.07447633e-01
    7.98475221e-02 -5.55305593e-02  3.17275733e-01  1.44845158e-01
    1.24596432e-01 -1.28748730e-01  2.49188855e-01  2.42388353e-01
   -1.43660501e-01  2.05613181e-01  8.00239295e-02  2.09122583e-01
    1.90970525e-01  1.24432109e-01 -5.74168488e-02  2.02242984e-03
    2.27665603e-01  1.06422693e-01  2.31825873e-01  1.54112145e-01
    1.78086638e-01  2.88183153e-01 -2.45962456e-01  1.02495201e-01
    2.35491410e-01 -3.01015913e-01 -2.90803492e-01  3.27953726e-01
   -4.02901143e-01  2.71164119e-01  2.30024695e-01 -4.16599624e-02
    3.59374970e-01 -3.41068119e-01  3.33613127e-01 -7.58210659e-01
   -7.37420544e-02 -3.69219147e-02  2.64651686e-01  3.49414229e-01
    1.83563888e-01  1.54265180e-01  2.72774808e-02  2.09104627e-01
   -1.23042911e-01  1.78791061e-01 -1.94657981e-01  3.31034929e

In [ ]:

tf.saved_model.save(model, full_model_dir)
hist_df = pd.DataFrame(history.history)
    
with open(os.path.join(modelpath, historyfile), mode="w") as f:
    hist_df.to_json(f)